# Homework 03: Python Fundamentals

This notebook completes the Stage 3 assignment using NumPy and pandas. It demonstrates elementwise operations, compares loop and vectorized execution, explores the provided dataset, calculates summary statistics, aggregates by category, and saves the results.

In [ ]:
from pathlib import Path
import timeit

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The course structure keeps homework notebooks at the homework root.
ROOT = Path.cwd()

from src.utils import get_summary_stats

DATA_PATH = ROOT / 'data' / 'raw' / 'starter_data.csv'
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. NumPy operations

NumPy applies arithmetic to every array element without requiring an explicit Python loop.

In [ ]:
numbers = np.arange(1, 11)
squared = numbers**2
scaled = numbers * 3 + 2

print('Original:', numbers)
print('Squared: ', squared)
print('3x + 2:  ', scaled)

In [ ]:
# Compare equivalent loop-based and vectorized calculations.
large_array = np.arange(100_000)
loop_result = np.array([value**2 for value in large_array])
vectorized_result = large_array**2
assert np.array_equal(loop_result, vectorized_result)

loop_time = timeit.timeit(lambda: [value**2 for value in large_array], number=10)
vectorized_time = timeit.timeit(lambda: large_array**2, number=10)

print(f'Loop time (10 runs):       {loop_time:.4f} seconds')
print(f'Vectorized time (10 runs): {vectorized_time:.4f} seconds')
print(f'Vectorization speedup:      {loop_time / vectorized_time:.1f}x')

The assertion confirms that both approaches produce identical values. The timing comparison shows that NumPy vectorization is faster because its elementwise work runs in optimized compiled code rather than a Python-level loop.

## 2. Dataset loading and inspection

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

In [ ]:
df.info()

The dataset has 10 complete rows, one categorical column (`category`), one numeric column (`value`), and one parsed date column (`date`).

## 3. Summary statistics and grouped aggregation

In [ ]:
# Use the imported reusable function to describe numeric columns.
summary = get_summary_stats(df)
summary

In [ ]:
# Compare the value distribution for each category.
category_summary = (
    df.groupby('category', as_index=False)
      .agg(count=('value', 'count'), mean=('value', 'mean'),
           minimum=('value', 'min'), maximum=('value', 'max'))
)
category_summary

Category C has the highest average value (about 27.7), while category A has the lowest (11.5). Category A appears four times; categories B and C each appear three times.

## 4. Save outputs

In [ ]:
summary_path = OUTPUT_DIR / 'summary.csv'
category_summary_path = OUTPUT_DIR / 'category_summary.csv'
summary.to_csv(summary_path)
category_summary.to_csv(category_summary_path, index=False)

print(f'Saved numeric summary to {summary_path}')
print(f'Saved category summary to {category_summary_path}')

### Bonus: basic plot

In [ ]:
plot_path = OUTPUT_DIR / 'average_value_by_category.png'

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(category_summary['category'], category_summary['mean'], color='#4C78A8')
ax.bar_label(bars, fmt='%.1f', padding=3)
ax.set(title='Average Value by Category', xlabel='Category', ylabel='Average value')
ax.set_ylim(0, category_summary['mean'].max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved plot to {plot_path}')

## Conclusion

The numeric values range from 10 to 30, with an overall mean of 17.6. Grouping reveals a clear difference among categories: C has the largest average, followed by B and then A. The analysis is reproducible because the summary logic lives in `src/utils.py` and all generated artifacts are saved under `data/processed/`.